# APD Sentinel — latih model APD (helm, rompi, sepatu, orang)

Jalankan di **Google Colab** dengan GPU: menu *Runtime → Change runtime type → T4 GPU*, lalu
*Runtime → Run all*. Waktu ±20–30 menit. Hasil: `ppe_project.pt` (dan `ppe_project.onnx`).

1. Data: dataset resmi Ultralytics **construction-ppe** (±1.400 foto, AGPL-3.0), kelas dipetakan
   ke 4 kelas proyek: `helmet, vest, shoes, person`.
2. **Opsional, paling penting untuk akurasi:** upload `workshop.zip` berisi foto bengkel sendiri
   (`images/*.jpg` + `labels/*.txt`, format YOLO, kelas sama) dari `scripts/collect_workshop.py`.
3. Model awal: YOLOv8n helm/rompi (Hansung-Cho, MIT) → fine-tune.
4. Unduh `ppe_project.pt`, taruh di `APDChecker/models/`. `vision.py` otomatis memakainya.
   Bandingkan dulu dengan `scripts/eval_ppe.py`; pakai hanya kalau lebih baik.

In [ ]:
!pip -q install ultralytics==8.4.175 huggingface_hub
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA - ganti runtime ke GPU")

In [ ]:
import os, shutil, zipfile, random, urllib.request
from pathlib import Path
W = Path("/content/work"); W.mkdir(exist_ok=True)
z = W / "construction-ppe.zip"
if not z.exists():
    urllib.request.urlretrieve("https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip", z)
src = W / "construction-ppe"
if not src.exists():
    zipfile.ZipFile(z).extractall(src)
print(sorted(p.name for p in src.iterdir()))

Opsional: upload `workshop.zip` (lewati sel ini kalau belum punya foto bengkel).

In [ ]:
from google.colab import files
up = files.upload()            # pilih workshop.zip
for name in up:
    zipfile.ZipFile(name).extractall(W / "workshop")
print(list((W / "workshop").rglob("*.jpg"))[:3])

In [ ]:
CLASSES = ("helmet", "vest", "shoes", "person")
CPPE = {0: 0, 2: 1, 3: 2, 6: 3}       # helmet, vest, boots, Person (others dropped)
data = W / "ppe-project"
shutil.rmtree(data, ignore_errors=True)
for split in ("train", "val", "test"):
    (data / "images" / split).mkdir(parents=True); (data / "labels" / split).mkdir(parents=True)
    for img in (src / "images" / split).iterdir():
        lab = src / "labels" / split / (img.stem + ".txt"); rows = []
        if lab.exists():
            for line in lab.read_text().splitlines():
                p = line.split()
                if len(p) >= 5 and int(p[0]) in CPPE:
                    rows.append(" ".join([str(CPPE[int(p[0])])] + p[1:5]))
        shutil.copy2(img, data / "images" / split / img.name)
        (data / "labels" / split / (img.stem + ".txt")).write_text("
".join(rows) + ("
" if rows else ""))
ws = sorted((W / "workshop").rglob("*.jpg")) if (W / "workshop").exists() else []
random.Random(0).shuffle(ws)
n_val = max(1, int(0.15 * len(ws))) if ws else 0
for k, img in enumerate(ws):
    lab = next(iter(img.parent.parent.rglob(img.stem + ".txt")), None)
    if lab is None: continue
    split, copies = ("val", 1) if k < n_val else ("train", 3)   # workshop photos weighted x3
    for r in range(copies):
        shutil.copy2(img, data / "images" / split / f"{img.stem}_r{r}.jpg")
        shutil.copy2(lab, data / "labels" / split / f"{img.stem}_r{r}.txt")
(data / "data.yaml").write_text(f"path: {data}
train: images/train
val: images/val
test: images/test
"
                                 "names:
" + "".join(f"  {i}: {c}
" for i, c in enumerate(CLASSES)))
print("foto bengkel:", len(ws)); print((data / "data.yaml").read_text())

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
base = hf_hub_download("Hansung-Cho/yolov8-ppe-detection", "best.pt")
model = YOLO(base)
model.train(data=str(data / "data.yaml"), epochs=80, imgsz=640, batch=32, device=0, patience=20,
            cos_lr=True, close_mosaic=10, hsv_v=0.5, degrees=10, project="/content/runs", name="ppe", exist_ok=True)

In [ ]:
best = YOLO("/content/runs/ppe/weights/best.pt")
for sz in (640, 320):
    m = best.val(data=str(data / "data.yaml"), split="test", imgsz=sz, verbose=False)
    print(f"imgsz {sz}: mAP50 {m.box.map50:.3f}  per kelas", {c: round(v, 3) for c, v in zip(CLASSES, m.box.ap50)})

In [ ]:
from google.colab import files
shutil.copy("/content/runs/ppe/weights/best.pt", "/content/ppe_project.pt")
YOLO("/content/ppe_project.pt").export(format="onnx", imgsz=320, opset=12, simplify=True)   # untuk NPU Pico nanti
files.download("/content/ppe_project.pt")